# Optuna search for Adam hyperparameters

Searches for the best Adam optimizer hyperparameters for
`optimize_multiple_radially_convex_sets_with_movable_circles`.
Loss-term weights are design parameters fixed at their defaults.

**Searched hyperparameters:**

| Parameter | Range | Scale | Adam default |
|---|---|---|---|
| `learning_rate` | 1e-4 – 5e-2 | log | 0.001 |
| `b1` | 0.5 – 0.99 | linear | 0.9 |
| `b2` | 0.9 – 0.9999 | linear | 0.999 |

**Quality score** (lower is better): sum of raw (unweighted) constraint violations —
enclosure + exclusion + circle collision — each divided by its fixed weight,
plus a small area contribution (×0.05) as a compactness tiebreaker.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import pandas as pd
import optuna

from vizopt.templates.stars_vs_circles import optimize_multiple_radially_convex_sets_with_movable_circles
from vizopt.components.stars import Discrete
from vizopt.base import OptimConfig

optuna.logging.set_verbosity(optuna.logging.WARNING)

In [ ]:
# Problem: 4 sets of 6 circles, randomly scattered
rng = np.random.default_rng(42)

N_SETS = 4
N_PER_SET = 6
N = N_SETS * N_PER_SET

positions = rng.uniform(-4.0, 4.0, size=(N, 2)).astype(np.float32)
radii = rng.uniform(0.25, 0.55, size=(N, 1)).astype(np.float32)
circles = np.hstack([positions, radii])

sets = [
    list(range(s * N_PER_SET, (s + 1) * N_PER_SET))
    for s in range(N_SETS)
]

# Fixed design parameters — not part of the search
FIXED_WEIGHTS = dict(
    weight_area=1.0,
    weight_perimeter=1.0,
    weight_exclusion=10.0,
    weight_smoothness=1.0,
    weight_position_anchor=1.0,
    weight_circle_collision=10.0,
    weight_bounding_box=0.0,
    weight_set_attraction=0.0,
    circle_collision_alpha=0.0,
)

# enclosure weight is hardcoded to 10.0 inside the template
_ENCLOSURE_W = 10.0

# coarser representation during search
SEARCH_K_ANGLES = 32
SEARCH_N_ITERS = 1500

print(f"N circles: {N}, N sets: {N_SETS}")

In [ ]:
def run_optimization(adam_params, n_iters=SEARCH_N_ITERS, k_angles=SEARCH_K_ANGLES):
    """Run with fixed design weights and the given Adam params; return history."""
    _, _, history, _ = optimize_multiple_radially_convex_sets_with_movable_circles(
        circles,
        sets,
        representation=Discrete(k_angles=k_angles),
        **FIXED_WEIGHTS,
        optim_config=OptimConfig(
            n_iters=n_iters,
            learning_rate=adam_params["learning_rate"],
            b1=adam_params["b1"],
            b2=adam_params["b2"],
        ),
    )
    return history


def score_from_history(history):
    """Quality score from the final history entry (lower is better).

    Raw (unweighted) constraint violations independent of the searched params.
    """
    final = history[-1]
    raw_encl = final["enclosure"] / _ENCLOSURE_W
    raw_excl = final["exclusion"] / FIXED_WEIGHTS["weight_exclusion"]
    raw_coll = final["circle_collision"] / FIXED_WEIGHTS["weight_circle_collision"]
    raw_area = final["area"] / FIXED_WEIGHTS["weight_area"]
    return float(raw_encl + raw_excl + raw_coll + 0.05 * raw_area)

## Baseline: Adam defaults

In [ ]:
DEFAULT_ADAM = {"learning_rate": 0.001, "b1": 0.9, "b2": 0.999}

history_baseline = run_optimization(DEFAULT_ADAM)
score_baseline = score_from_history(history_baseline)
print(f"Baseline score: {score_baseline:.4f}")

## Optuna hyperparameter search

In [ ]:
N_TRIALS = 60


def objective(trial):
    adam_params = {
        "learning_rate": trial.suggest_float("learning_rate", 1e-4, 5e-2, log=True),
        "b1": trial.suggest_float("b1", 0.5, 0.99),
        "b2": trial.suggest_float("b2", 0.9, 0.9999),
    }
    return score_from_history(run_optimization(adam_params))


study = optuna.create_study(
    direction="minimize",
    sampler=optuna.samplers.TPESampler(seed=0),
)
study.enqueue_trial(DEFAULT_ADAM)
study.optimize(objective, n_trials=N_TRIALS, show_progress_bar=True)

best = study.best_trial
print(f"\nBaseline score: {study.trials[0].value:.4f}")
print(f"Best score:     {best.value:.4f}  (trial #{best.number})")
print(f"Improvement:    {(study.trials[0].value - best.value) / study.trials[0].value * 100:.1f}%")
print("\nBest params:")
for k, v in best.params.items():
    print(f"  {k}: {v:.4g}")

## Results analysis

In [ ]:
trial_scores = [t.value for t in study.trials if t.value is not None]
running_best = np.minimum.accumulate(trial_scores)

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(trial_scores, "o", alpha=0.5, ms=4, label="trial score")
ax.plot(running_best, lw=2, label="running best")
ax.axhline(trial_scores[0], ls="--", color="grey", label="baseline (Adam defaults)")
ax.set_xlabel("trial")
ax.set_ylabel("score (lower is better)")
ax.set_title("Trial scores")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
df = pd.DataFrame([t.params | {"score": t.value} for t in study.trials if t.value is not None])
param_names = list(best.params.keys())
threshold_20pct = np.percentile(df["score"], 20)

fig, axes = plt.subplots(1, 3, figsize=(13, 4))
log_params = {"learning_rate"}

for ax, name in zip(axes, param_names):
    x = df[name]
    y = df["score"]
    is_top = y <= threshold_20pct
    if name in log_params:
        ax.set_xscale("log")
    ax.scatter(x[~is_top], y[~is_top], alpha=0.4, s=20, color="steelblue")
    ax.scatter(x[is_top], y[is_top], alpha=0.8, s=30, color="orange", label="top 20%")
    ax.axvline(best.params[name], ls="--", color="red", lw=1.2, label="best")
    ax.axvline(DEFAULT_ADAM[name], ls=":", color="grey", lw=1.2, label="default")
    ax.set_xlabel(name, fontsize=9)
    ax.set_ylabel("score", fontsize=9)

axes[0].legend(fontsize=8)
fig.suptitle("Adam param vs score  (orange = top 20%, red = best, grey = default)", fontsize=10)
plt.tight_layout()
plt.show()

In [ ]:
try:
    importances = optuna.importance.get_param_importances(study)
    fig, ax = plt.subplots(figsize=(5, 3))
    names = list(importances.keys())
    vals = list(importances.values())
    ax.barh(names[::-1], vals[::-1], color="steelblue")
    ax.set_xlabel("importance")
    ax.set_title("Hyperparameter importance (fANOVA)")
    plt.tight_layout()
    plt.show()
except Exception as e:
    print(f"Importance computation skipped: {e}")

## Full-resolution run with best parameters

In [ ]:
FINAL_K_ANGLES = 64
FINAL_N_ITERS = 3000


def full_run(adam_params):
    results, circles_out, history, _ = optimize_multiple_radially_convex_sets_with_movable_circles(
        circles,
        sets,
        representation=Discrete(k_angles=FINAL_K_ANGLES),
        **FIXED_WEIGHTS,
        optim_config=OptimConfig(
            n_iters=FINAL_N_ITERS,
            learning_rate=adam_params["learning_rate"],
            b1=adam_params["b1"],
            b2=adam_params["b2"],
        ),
    )
    return results, circles_out, history


results_default, circles_default, history_default = full_run(DEFAULT_ADAM)
results_best, circles_best, history_best = full_run(best.params)

score_default_full = score_from_history(history_default)
score_best_full = score_from_history(history_best)

print(f"Default Adam score (full run): {score_default_full:.4f}")
print(f"Best Adam score   (full run): {score_best_full:.4f}")
print(f"Improvement: {(score_default_full - score_best_full) / score_default_full * 100:.1f}%")

In [ ]:
term_names = ["enclosure", "exclusion", "circle_collision", "area", "perimeter", "smoothness"]

fig, axes = plt.subplots(2, 3, figsize=(14, 7), sharex=True)
axes = axes.flatten()

for ax, term in zip(axes, term_names):
    iters_d = [r["iteration"] for r in history_default]
    vals_d = [r.get(term, float("nan")) for r in history_default]
    iters_b = [r["iteration"] for r in history_best]
    vals_b = [r.get(term, float("nan")) for r in history_best]
    ax.plot(iters_d, vals_d, label="default Adam", color="steelblue")
    ax.plot(iters_b, vals_b, label="best Adam", color="orange")
    ax.set_title(term)
    ax.set_xlabel("iteration")
    ax.set_ylabel("weighted loss")
    ax.legend(fontsize=8)

plt.suptitle("Loss curves: default Adam vs best Adam hyperparameters", fontsize=11)
plt.tight_layout()
plt.show()

In [ ]:
def plot_layout(ax, results, circles_out, title):
    cmap = cm.get_cmap("tab10", N_SETS)
    ax.set_aspect("equal")
    ax.axis("off")
    ax.set_title(title, fontsize=10)
    for s, res in enumerate(results):
        color = cmap(s)
        cx, cy = res["center"]
        angs, rads = res["angles"], res["radii"]
        bx = np.append(cx + rads * np.cos(angs), cx + rads[0] * np.cos(angs[0]))
        by = np.append(cy + rads * np.sin(angs), cy + rads[0] * np.sin(angs[0]))
        ax.fill(bx, by, alpha=0.15, color=color)
        ax.plot(bx, by, color=color, lw=1.5)
    for s in range(N_SETS):
        color = cmap(s)
        for i in sets[s]:
            cx_c, cy_c, r = circles_out[i]
            ax.add_patch(plt.Circle((cx_c, cy_c), r, color=color, alpha=0.45,
                                    linewidth=0.8, edgecolor=color))
    ax.autoscale_view()


fig, axes = plt.subplots(1, 2, figsize=(14, 7))
plot_layout(axes[0], results_default, circles_default,
            f"Default Adam  lr={DEFAULT_ADAM['learning_rate']}, b1={DEFAULT_ADAM['b1']}, b2={DEFAULT_ADAM['b2']}\n"
            f"score: {score_default_full:.3f}")
plot_layout(axes[1], results_best, circles_best,
            f"Best Adam  lr={best.params['learning_rate']:.3g}, b1={best.params['b1']:.3f}, b2={best.params['b2']:.4f}\n"
            f"score: {score_best_full:.3f}")
plt.tight_layout()
plt.show()

In [ ]:
rows = [
    {"parameter": k, "default": DEFAULT_ADAM[k], "best": best.params[k]}
    for k in param_names
]
pd.DataFrame(rows).set_index("parameter").style.format("{:.4g}")